In [0]:
%skip
%pip install openpyxl==3.1.2 "mlflow[databricks]" "pydantic>=1.10,<2"

In [0]:
%skip
import pyspark.sql.functions as fn
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import KBinsDiscretizer, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import mlflow
import numpy as np
import pandas as pd

In [0]:
%sql
create or replace temp view gamesum as (
    select * from hbse.dbt_marts.fct_njd_game_summary_enhanced where accountid is not null
);

create or replace temp view clicks as (
    select c.audienceid, eventdate, sendid from kagr_njd.stage.sfmcclicks a
    join kagr_njd.stage.rawaudience b on a.subscriberkey = b.SOURCEACCOUNTID
    join kagr_njd.stage.audiencemapping c on b.RAWAUDIENCEID = c.RAWAUDIENCEID
);

create or replace temp view audienceids as (
    select distinct audienceid from kagr_njd.stage.audience
);


In [0]:
transactions = spark.table("gamesum")
clicks = spark.table("clicks")
audienceids = spark.table("audienceids")

display(clicks)

In [0]:
def score_fans(audience_df, digital_df, transactions_df, dynamic_bins=True,
               recency_bins=[-1, 484, 612, 900, 1210, np.inf],
               monetary_bins=[-1, 160, 348, 660, 1460, np.inf]):

  txns = (
    transactions_df
     .withColumn('total_revenue', fn.col('purchaseprice'))
      .filter('total_revenue > 0.0')
      .withColumn('saledate', fn.to_date('saledate'))
      .withColumn('eventdate', fn.to_date('eventdate'))
      .groupBy('audienceid')
        .agg(
          fn.sum('total_revenue').alias('total_revenue'),
          fn.max('saledate').alias('lastdate'),
          fn.countDistinct('eventdate').alias('frequency'),
          fn.avg('total_revenue').alias('atp'),
          fn.min(fn.datediff(fn.current_date(), 'saledate')).alias('recency')
        )
    )

  rfm_capped = (
    txns
      .withColumn('frequency', fn.expr("case when frequency > 5 then 5 else frequency end"))
      .withColumn('atp', fn.expr("case when atp > 5000 then 5000 else atp end"))
    )

  scored_pd = rfm_capped.toPandas()

  # recency_score
  if dynamic_bins:
    _, recency_edges = pd.qcut(scored_pd['recency'], 5, retbins=True, duplicates='drop')
    recency_edges[0], recency_edges[-1] = -1, np.inf
  else:
    recency_edges = recency_bins
  recency_labels = list(range(len(recency_edges) - 1, 0, -1))  # fewer days = higher score
  recency_score = pd.cut(scored_pd['recency'], bins=recency_edges, labels=recency_labels).astype(int)

  # monetary_score
  if dynamic_bins:
    _, monetary_edges = pd.qcut(scored_pd['atp'], 5, retbins=True, duplicates='drop')
    monetary_edges[0], monetary_edges[-1] = -1, np.inf
  else:
    monetary_edges = monetary_bins
  monetary_labels = list(range(1, len(monetary_edges)))
  monetary_score = pd.cut(scored_pd['atp'], bins=monetary_edges, labels=monetary_labels).astype(int)

  # frequency_score: value-based bins 1-5 (capped at 5)
  frequency_score = scored_pd['frequency'].clip(upper=5).astype(int)

  scored_pd['recency_score'] = recency_score
  scored_pd['frequency_score'] = frequency_score
  scored_pd['monetary_score'] = monetary_score
  scored_pd['aboslute_score'] = recency_score + frequency_score + monetary_score
  scored_pd['composite_score'] = (recency_score*3) + (frequency_score*2) + monetary_score

  for col in ['recency_score', 'frequency_score', 'monetary_score', 'aboslute_score']:
    scored_pd[f'{col}_zscore'] = ((scored_pd[col] - scored_pd[col].mean()) / scored_pd[col].std()).round(4)

  # merge back to full audience so every audienceid is represented (even those without transactions)
  audience_pd = audience_df.toPandas()
  audience_pd['audienceid'] = audience_pd['audienceid'].astype(str)
  scored_pd['audienceid'] = scored_pd['audienceid'].astype(str)

  result_pd = audience_pd.merge(scored_pd, on='audienceid', how='left')
  score_cols = [c for c in scored_pd.columns if c != 'audienceid']
  fill_cols = [c for c in score_cols if c != 'lastdate']
  result_pd[fill_cols] = result_pd[fill_cols].fillna(0)

  clicks_agg_pd = (
    digital_df
      .groupBy('audienceid')
      .agg(fn.count('*').alias('total_clicks'))
      .toPandas()
  )
  clicks_agg_pd['audienceid'] = clicks_agg_pd['audienceid'].astype(str)

  result_pd = result_pd.merge(clicks_agg_pd, on='audienceid', how='left')
  result_pd['total_clicks'] = result_pd['total_clicks'].fillna(0)

  # fans with 0 clicks get score 1; the rest are binned into quintiles
  result_pd['click_score'] = pd.qcut(
    result_pd['total_clicks'].rank(method='first'),
    q=5, labels=[1, 2, 3, 4, 5]
  ).astype(int)

  return result_pd[['audienceid','aboslute_score', 'composite_score','recency_score', 'frequency_score', 'monetary_score', 'click_score']]

# Replace table here**
new_scores_pd = score_fans(audience_df=audienceids, digital_df=clicks, transactions_df=transactions)
display(new_scores_pd)

In [0]:
print(f"Total rows: {len(new_scores_pd):,}")
print(f"Unique audienceids: {new_scores_pd['audienceid'].nunique():,}")
print(f"Rows with aboslute_score = 0: {(new_scores_pd['aboslute_score'] == 0).sum():,}")

In [0]:
sent_pd = (
  spark.sql("""
    SELECT
      g.audienceid,
      COUNT(*) AS total_sent
    	FROM kagr_njd.stage.sfmcsendjobs a
	  JOIN kagr_njd.stage.sfmcsent b on a.sendid = b.sendid
    JOIN kagr_njd.stage.rawaudience f ON b.subscriberkey = f.sourceaccountid
    JOIN kagr_njd.stage.audiencemapping g ON f.rawaudienceid = g.rawaudienceid
    WHERE a.subject NOT LIKE '%Test%' AND a.subject NOT LIKE '%test%'and a.emailname LIKE '%Devils%'
    GROUP BY g.audienceid
""")
  .toPandas()
)



agg_fields_pd['audienceid'] = agg_fields_pd['audienceid'].astype(str)
sent_pd['audienceid'] = sent_pd['audienceid'].astype(str)
clicks_pd['audienceid'] = clicks_pd['audienceid'].astype(str)
sales_funnel['audienceid'] = sales_funnel['audienceid'].astype(str)
new_scores_pd['audienceid'] = new_scores_pd['audienceid'].astype(str)

full_fan_with_agg = (
  new_scores_pd
    .merge(agg_fields_pd, on='audienceid', how='left')
    .merge(clicks_pd, on='audienceid', how='left')
    .merge(sent_pd, on='audienceid', how='left')
    .merge(sales_funnel, on='audienceid', how='left')
  )
full_fan_with_agg['sales_funnel'] = full_fan_with_agg['sales_funnel'].fillna('Null')
full_fan_with_agg = full_fan_with_agg.fillna(0)

full_fan_with_agg['click_rate'] = (full_fan_with_agg['total_clicks'] / full_fan_with_agg['total_sent']).replace([float('inf')], 0).fillna(0).round(4)

display(full_fan_with_agg)

In [0]:
%skip
table_name = "hbse.default.njd_rfm_dataset" 

primary_rev = (
  spark.table(table_name)
      .withColumn('total_revenue', fn.round(fn.col('total_sales') * fn.col('total_seats'), 2)) 
  )

display(primary_rev)

In [0]:
%skip
primary_rev_consolidated = (
primary_rev
    .filter('total_revenue > 0.0') # remove returns from dataset
    .withColumn('saledate', fn.to_date('saledate')) # convert date to just datepart
    .groupBy('audienceid', 'saledate', 'ledgername') # group on customer and date
      .agg(fn.sum('total_revenue').alias('total_revenue')) # sum sales amount
  )

display(primary_rev_consolidated)

In [0]:
%skip
# get last date in dataset
last_date = (
  primary_rev_consolidated
    .groupBy()
      .agg(fn.max('saledate').alias('lastdate'))
  )

# calculate metrics
rfm_metrics = (
  primary_rev_consolidated
    .crossJoin(last_date)
    .groupBy('audienceid') # for each customer
      .agg(
        fn.min(fn.datediff('lastdate','saledate')).alias('recency'), # days since last date in dataset (get lowest value as recency)
        fn.countDistinct('saledate').alias('frequency'), # unique dates on which purchases occur
        fn.round(fn.avg('total_revenue'), 2).alias('monetary_value') # avg spend per purchase, rounded to 2 decimal places (works because we've already summed sales per customer-date)
        )
  )

# display metrics
display(
  rfm_metrics
)

In [0]:
%skip
df_pd = rfm_metrics.toPandas()

display(df_pd[['recency', 'frequency', 'monetary_value']].quantile([.1, .25, .5, .75, .9, 1.0]))

f, axes = plt.subplots(nrows=1, ncols=3, squeeze=True, figsize=(32,10))

for i, metric in enumerate(['recency', 'frequency', 'monetary_value']):
   
  axes[i].set_title(metric)
  
  axes[i].hist(df_pd[metric], bins=10)

In [0]:
%skip
print('monetary_value high percentiles:')
print(df_pd['monetary_value'].describe(percentiles=[.75, .9, .95, .975, .99, .995, .999]))

pct_at_or_below_3000 = (df_pd['monetary_value'] <= 3000).mean() * 100
print(f"\nthe current $3000 cap sits at the {pct_at_or_below_3000:.1f}th percentile "
      f"({100 - pct_at_or_below_3000:.1f}% of customers get capped)")

print('\ncustomers affected by candidate caps:')
for cap in [1500, 3000, 5000, 10000, 20000, 50000, 100000]:
  n_above = (df_pd['monetary_value'] > cap).sum()
  pct_above = n_above / len(df_pd) * 100
  print(f'  cap=${cap:>7,}: {n_above:>5} customers above ({pct_above:.2f}%)')

In [0]:
%skip
rfm_metrics_cleansed = (
  rfm_metrics
    .withColumn('frequency', fn.expr("case when frequency > 5 then 5 else frequency end"))
    .withColumn('monetary_value', fn.expr("case when monetary_value > 5000 then 5000 else monetary_value end"))
  )

In [0]:
%skip
df_pd = rfm_metrics_cleansed.toPandas()

f, axes = plt.subplots(nrows=1, ncols=3, squeeze=True, figsize=(32,10))

for i, metric in enumerate(['recency', 'frequency', 'monetary_value']):
   
  axes[i].set_title(metric)
  
  axes[i].hist(df_pd[metric], bins=10)

In [0]:
%skip
inputs_pd = rfm_metrics_cleansed.toPandas()

In [0]:
%skip
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

# apply binner to recency and monetary only; frequency is binary encoded in the next cell (92% of customers have frequency=1)
col_trans = ColumnTransformer(
  [
    ('r_bin', binner, ['recency']),
    ('m_bin', binner, ['monetary_value'])
    ],
  remainder='drop'
  )

In [0]:
%skip
# invert the recency values so that higher is better
inputs_pd['recency'] = inputs_pd['recency'] * -1

bins = col_trans.fit_transform(inputs_pd)

inputs_pd['r_bin'] = bins[:,0]
inputs_pd['m_bin'] = bins[:,1]

inputs_pd['f_bin'] = (inputs_pd['frequency'] > 1).astype(float)

display(inputs_pd)

In [0]:
%skip
# recency (after -1 inversion): unique values + top value counts
print(f"Unique recency values: {inputs_pd['recency'].nunique()}")
print(f"Total customers: {len(inputs_pd)}")
print(f"\nRecency value counts (top 20):")
print(inputs_pd['recency'].value_counts().head(20).to_string())

# frequency: unique values, value counts, and the resulting f_bin distribution
print(f"\nUnique frequency values: {inputs_pd['frequency'].nunique()}")
print(f"\nFrequency value counts (top 20):")
print(inputs_pd['frequency'].value_counts().head(20).to_string())
print(f"\nf_bin distribution:")
print(inputs_pd['f_bin'].value_counts().sort_index().to_string())

# recency histogram + describe
counts, edges = np.histogram(inputs_pd['recency'], bins=10)
print("\nRecency histogram (raw values currently in inputs_pd):")
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"  [{lo:8.1f}, {hi:8.1f}) -> {c}")

print("\nrecency describe:")
print(inputs_pd['recency'].describe())

sale_activity = (
  primary_rev_consolidated
    .groupBy(fn.date_trunc('MONTH', 'saledate').alias('month'))
    .agg(fn.count('*').alias('n_transactions'))
    .orderBy('month')
  )
display(sale_activity)

In [0]:
%skip
f, axes = plt.subplots(nrows=1, ncols=3, squeeze=True, figsize=(32,10))

for i, metric in enumerate(['r_bin','f_bin','m_bin']):
   
  axes[i].set_title(metric)
  
  axes[i].hist(inputs_pd[metric], bins=5)

In [0]:
%skip
for bin_col, metric in [('r_bin', 'recency'), ('f_bin', 'frequency'), ('m_bin', 'monetary_value')]:

  df = inputs_pd[[bin_col, metric]].copy()

  if metric == 'recency':
    df[metric] = df[metric].abs()

  ranges = (
    df
      .groupby(bin_col)[metric]
      .agg(min_value='min', max_value='max', customers='count')
      .sort_index()
      .reset_index()
    )

  print(f"\n{bin_col} ranges (based on {metric}):")
  display(ranges)

In [0]:
%skip
tsne = TSNE(n_components=2, perplexity=80, init='pca', learning_rate='auto')
tsne_results = tsne.fit_transform(inputs_pd[['r_bin','f_bin','m_bin']])

inputs_pd['tsne_one'] = tsne_results[:,0]
inputs_pd['tsne_two'] = tsne_results[:,1]

display(inputs_pd)

In [0]:
%skip
f, axes = plt.subplots(nrows=1, ncols=3, squeeze=True, figsize=(32,10))

for i, metric in enumerate(['r_bin', 'f_bin', 'm_bin']):
  
  n = inputs_pd[['{0}'.format(metric)]].nunique()[0]
  
  axes[i].set_title(metric)
  
  sns.scatterplot(
    x='tsne_one',
    y='tsne_two',
    hue='{0}'.format(metric),
    palette=sns.color_palette('coolwarm', n),
    data=inputs_pd,
    legend=False,
    alpha=0.4,
    ax = axes[i]
    )

In [0]:
%skip
max_k = 15

_inputs_data = inputs_pd[['r_bin','f_bin','m_bin']].copy()

# function to train and score clusters based on k cluster count
@fn.udf('float')
def get_silhouette(k):

  km = KMeans(
    n_clusters=k, 
    init='random',
    n_init=10000
    )
  kmeans = km.fit( _inputs_data )

  silhouette = silhouette_score( 
      _inputs_data,  # x values
      kmeans.predict(_inputs_data), # cluster assignments 
      sample_size=2000
      )
  
  return float(silhouette)


iterations = (
  spark
    .range(2, max_k + 1, step=1) # get values for k
    .withColumnRenamed('id','k') # rename to k
    .repartition( max_k-1, 'k' ) # ensure data are well distributed
    .withColumn('silhouette', get_silhouette('k'))
  )
  
display(iterations)

Databricks visualization. Run in Databricks to view.

In [0]:
%skip
model = KMeans(
  n_clusters=4, 
  init='random',
  n_init=10000
  )

pipe = Pipeline(steps=[
  ('binnerize', col_trans),
  ('cluster', model)
  ])

fitted_pipe = pipe.fit( inputs_pd )

inputs_pd['cluster'] = pipe.predict( inputs_pd )

display(inputs_pd)

In [0]:
%skip
# composite RFM score: weighted combination of recency, frequency, and monetary value
# recency_days is only used to restore a positive, human-readable value in the displayed table.

recency_days = inputs_pd['recency'].abs()

composite = round((recency_days * 3) + (inputs_pd['frequency'] * 2) + (inputs_pd['monetary_value'] * 1), 2)

inputs_pd['composite'] = composite
display(
  inputs_pd[['audienceid', 'recency', 'frequency', 'monetary_value', 'composite', 'cluster']]
    .assign(recency=recency_days)  # restore positive recency for readability
  )

In [0]:
%skip
print('recency_days quantiles (days since last purchase):')
print(recency_days.describe(percentiles=[.1, .25, .5, .75, .9]))

print('\nrecency_days quintile edges (for recalibrating the fixed bins below):')
_, quintile_edges = pd.qcut(recency_days, 5, retbins=True, duplicates='drop')
print([round(e) for e in quintile_edges])
print('\nrecency_score distribution (recalibrated fixed bins based on quantiles):')
print((inputs_pd['recency_score'].value_counts(normalize=True).sort_index() * 100).round(1).astype(str) + '%')

print('\n\nmonetary_value quantiles (avg spend per purchase, capped above):')
print(inputs_pd['monetary_value'].describe(percentiles=[.1, .25, .5, .75, .9]))
print('\nmonetary_value quintile edges (for recalibrating the fixed bins below):')
_, m_quintile_edges = pd.qcut(inputs_pd['monetary_value'], 5, retbins=True, duplicates='drop')
print([round(e, 2) for e in m_quintile_edges])
print('\nmonetary_score distribution (fixed bins:')
print((inputs_pd['monetary_score'].value_counts(normalize=True).sort_index() * 100).round(1).astype(str) + '%')

print('\n\nfrequency_score distribution (raw frequency capped at 5, used directly as score):')
print((inputs_pd['frequency_score'].value_counts(normalize=True).sort_index() * 100).round(1).astype(str) + '%')

In [0]:
%skip
#thresholds are set based on quintile values in previous cell
recency_score = pd.cut(
  recency_days,
  bins=[-1, 484, 612, 900, 1210, np.inf],
  labels=[5, 4, 3, 2, 1]
  ).astype(int)

frequency_score = inputs_pd['frequency'].clip(upper=5).astype(int)

monetary_score = pd.cut(
  inputs_pd['monetary_value'],
  bins=[-1, 160, 348, 660, 1460, np.inf],
  labels=[1, 2, 3, 4, 5]
  ).astype(int)


inputs_pd['recency_score'] = recency_score
inputs_pd['frequency_score'] = frequency_score
inputs_pd['monetary_score'] = monetary_score
inputs_pd['rfm_score_absolute'] = recency_score + frequency_score + monetary_score

display(
  inputs_pd[['audienceid', 'recency', 'frequency', 'monetary_value',
             'recency_score', 'frequency_score', 'monetary_score', 'rfm_score_absolute', 'composite', 'cluster']]
    .assign(recency=recency_days)
  )

In [0]:
%skip
f, axes = plt.subplots(nrows=1, ncols=4, squeeze=True, figsize=(42, 10))

axes[0].set_title('cluster')
sns.scatterplot(
  x='tsne_one',
  y='tsne_two',
  hue='cluster',
  palette=sns.color_palette('husl', inputs_pd[['cluster']].nunique()[0]),
  data=inputs_pd,
  alpha=0.4,
  ax = axes[0]
  )
axes[0].legend(loc='lower left', ncol=2, fancybox=True)

for i, metric in enumerate(['r_bin', 'f_bin', 'm_bin']):
  
  n = inputs_pd[['{0}'.format(metric)]].nunique()[0]
  
  axes[i+1].set_title(metric)
  
  sns.scatterplot(
    x='tsne_one',
    y='tsne_two',
    hue='{0}'.format(metric),
    palette=sns.color_palette('coolwarm', n),
    data=inputs_pd,
    legend=False,
    alpha=0.4,
    ax = axes[i+1]
    )

In [0]:
%skip
clusters = []

for c in range(0, pipe[-1].n_clusters):
  centroids = np.abs(pipe[-1].cluster_centers_[c].round(0).astype('int')).tolist()
  clusters += [ [c] + centroids]

clusters_pd = pd.DataFrame(clusters, columns=['cluster','r_bin','m_bin'])

display(clusters_pd)

In [0]:
%skip
r_median = clusters_pd['r_bin'].median()
m_median = clusters_pd['m_bin'].median()

def assign_label(row):
  is_recent = row['r_bin'] >= r_median
  is_high_value = row['m_bin'] >= m_median
  if is_recent and is_high_value:
    return 'Loyal High-Value'       # recent, high spenders — best active segment
  elif is_recent and not is_high_value:
    return 'Recent / Low-Value'     # fairly recent but lowest spenders — nurture to grow value
  elif not is_recent and is_high_value:
    return 'Lapsed High-Value'      # mostly lapsed but previously the biggest spenders — top win-back priority
  else:
    return 'Lost Customers'         # lapsed, low spenders — unlikely to return without intervention

clusters_pd['label'] = clusters_pd.apply(assign_label, axis=1)
display(clusters_pd[['cluster', 'r_bin', 'm_bin', 'label']])

In [0]:
%skip
customers_labeled = inputs_pd.merge(
  clusters_pd[['cluster', 'label']],
  on='cluster',
  how='left'
)

display(
  customers_labeled[['audienceid', 'recency', 'frequency', 'monetary_value', 'r_bin', 'f_bin', 'm_bin', 
  'recency_score', 'frequency_score', 'monetary_score', 'rfm_score_absolute', 'composite', 'cluster', 'label']]
  .assign(recency=customers_labeled['recency'].abs())  # restore positive recency for readability
  .sort_values('label')
  .reset_index(drop=True)
)

In [0]:
%skip
segment_counts = (
  customers_labeled
  .groupby(['cluster', 'label'])
  .agg(
    customers=('audienceid', 'count'),
    avg_recency=('recency', lambda x: x.abs().mean().round(0)),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary_value', 'mean')
  )
  .round({'avg_frequency': 2, 'avg_monetary': 2})
  .sort_values('cluster')
  .reset_index()
)
segment_counts['pct_of_total'] = (segment_counts['customers'] / segment_counts['customers'].sum() * 100).round(1).astype(str) + '%'

display(segment_counts[['cluster', 'label', 'customers', 'pct_of_total', 'avg_recency', 'avg_frequency', 'avg_monetary']])

In [0]:
%skip
order = clusters_pd.sort_values('cluster')['label'].tolist()

f, axes = plt.subplots(nrows=1, ncols=2, squeeze=True, figsize=(20, 8))

sns.boxplot(x='label', y='composite', data=customers_labeled, order=order, palette='viridis', ax=axes[0])
axes[0].set_title('Composite RFM Score by Cluster')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)

sns.boxplot(x='label', y='rfm_score_absolute', data=customers_labeled, order=order, palette='viridis', ax=axes[1])
axes[1].set_title('Absolute RFM Score by Cluster')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()

In [0]:
%skip
model_k6 = KMeans(
  n_clusters=6,
  init='random',
  n_init=10000
  )

pipe_k6 = Pipeline(steps=[
  ('binnerize', col_trans),
  ('cluster', model_k6)
  ])

fitted_pipe_k6 = pipe_k6.fit(inputs_pd)
inputs_pd['cluster_k6'] = pipe_k6.predict(inputs_pd)

display(inputs_pd[['audienceid', 'r_bin', 'f_bin', 'm_bin', 'cluster', 'cluster_k6']])

In [0]:
%skip
clusters_k6 = []
for c in range(0, pipe_k6[-1].n_clusters):
  centroids = np.abs(pipe_k6[-1].cluster_centers_[c].round(0).astype('int')).tolist()
  clusters_k6 += [[c] + centroids]

clusters_k6_pd = pd.DataFrame(clusters_k6, columns=['cluster', 'r_bin', 'm_bin'])

clusters_k6_pd['recency_tier'] = pd.qcut(
  clusters_k6_pd['r_bin'].rank(method='first'), q=3, labels=['Lapsed', 'Mid-Recency', 'Recent']
  )
m_median_k6 = clusters_k6_pd['m_bin'].median()
clusters_k6_pd['value_tier'] = np.where(clusters_k6_pd['m_bin'] >= m_median_k6, 'High-Value', 'Low-Value')
clusters_k6_pd['label'] = clusters_k6_pd['recency_tier'].astype(str) + ' / ' + clusters_k6_pd['value_tier']

display(clusters_k6_pd[['cluster', 'r_bin', 'm_bin', 'label']])

In [0]:
%skip
customers_k6 = inputs_pd.merge(
  clusters_k6_pd[['cluster', 'label']].rename(columns={'cluster': 'cluster_k6', 'label': 'label_k6'}),
  on='cluster_k6', how='left'
  )

comparison = customers_labeled[['audienceid', 'label']].rename(columns={'label': 'label_k4'}).merge(
  customers_k6[['audienceid', 'label_k6']], on='audienceid', how='left'
  )

print('How each k=4 segment splits across the k=6 segments:')
crosstab = pd.crosstab(comparison['label_k4'], comparison['label_k6'])
display(crosstab)

k6_segment_counts = (
  customers_k6
    .groupby(['cluster_k6', 'label_k6'])
    .agg(
      customers=('audienceid', 'count'),
      avg_recency=('recency', lambda x: x.abs().mean().round(0)),
      avg_frequency=('frequency', 'mean'),
      avg_monetary=('monetary_value', 'mean')
      )
    .round({'avg_frequency': 2, 'avg_monetary': 2})
    .reset_index()
    .sort_values('cluster_k6')
  )
k6_segment_counts['pct_of_total'] = (k6_segment_counts['customers'] / k6_segment_counts['customers'].sum() * 100).round(1).astype(str) + '%'

print('\nk=6 segment summary:')
display(k6_segment_counts[['cluster_k6', 'label_k6', 'customers', 'pct_of_total', 'avg_recency', 'avg_frequency', 'avg_monetary']])

In [0]:
%skip
# Unity Catalog requires a fully qualified 3-level name: catalog.schema.model_name
model_name = 'hbse.default.rfm_segmentation'

In [0]:
%skip
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
_ = mlflow.set_experiment('/Users/{}/{}'.format(username, model_name))

In [0]:
%skip
with mlflow.start_run(run_name='deployment ready'):

  sample_input = inputs_pd[['recency', 'frequency', 'monetary_value']].head(5)
  sample_output = fitted_pipe.predict(sample_input)
  signature = mlflow.models.infer_signature(sample_input, sample_output)

  mlflow.sklearn.log_model(
    fitted_pipe,
    'model',
    signature=signature,
    input_example=sample_input,
    registered_model_name=model_name
    )

In [0]:
%skip
client = mlflow.tracking.MlflowClient()

latest_model_info = client.search_model_versions(f"name='{model_name}'")[0]
model_version = latest_model_info.version

client.set_registered_model_alias(
  name=model_name,
  alias='production',
  version=model_version
  )

In [0]:
%skip
loaded_model = mlflow.sklearn.load_model(f'models:/{model_name}@production')

scored_pd = rfm_metrics_cleansed.toPandas()
scored_pd['cluster'] = loaded_model.predict(
  scored_pd[['recency', 'frequency', 'monetary_value']].assign(recency=lambda d: d['recency'] * -1)
  )

display(spark.createDataFrame(scored_pd))

In [0]:
%skip
agg_fields_pd = (
  spark.table('kagr_njd.stage.dim_aggregatefields')
    .select('audienceid', 'currentstm', 'priorstm', 'priorsgb')
    .toPandas()
  )

sent_pd = (
  spark.sql("""
    SELECT
      g.audienceid,
      COUNT(*) AS total_sent
    	FROM kagr_njd.stage.sfmcsendjobs a
	  JOIN kagr_njd.stage.sfmcsent b on a.sendid = b.sendid
    JOIN kagr_njd.stage.rawaudience f ON b.subscriberkey = f.sourceaccountid
    JOIN kagr_njd.stage.audiencemapping g ON f.rawaudienceid = g.rawaudienceid
    WHERE a.subject NOT LIKE '%Test%' AND a.subject NOT LIKE '%test%'and a.emailname LIKE '%Devils%'
    GROUP BY g.audienceid
""")
  .toPandas()
)

clicks_pd = (
  spark.sql("""
    SELECT
      g.audienceid,
      COUNT(*) AS total_clicks,
      COUNT(DISTINCT c.emailaddress) AS total_emailaddresses
    FROM kagr_njd.stage.sfmcclicks c
    JOIN kagr_njd.stage.rawaudience f ON c.subscriberkey = f.sourceaccountid
    JOIN kagr_njd.stage.audiencemapping g ON f.rawaudienceid = g.rawaudienceid
    GROUP BY g.audienceid
    """)
    .toPandas()
  )

sales_funnel = (
  spark.sql("""
    select audienceid, sales_funnel from (
      select distinct audienceid, upperfunnel, midfunnel, lowerfunnel, 
        CASE 
          WHEN (upperfunnel IN ('1') AND lowerfunnel IS NULL AND midfunnel IS NULL) then 'Upper'
          WHEN (midfunnel IN ('1') AND lowerfunnel IS NULL) then 'Mid'
          WHEN (lowerfunnel IN ('1')) then 'Lower'
          ELSE 'Null'
          END AS Sales_Funnel
      from KAGR_HBSE.NJD_FLYWHEEL_DATASETS.FLY_MAT_AUDIENCE
    )
    WHERE Sales_Funnel NOT IN ('Null')
    """)
  .toPandas()
)

agg_fields_pd['audienceid'] = agg_fields_pd['audienceid'].astype(str)
sent_pd['audienceid'] = sent_pd['audienceid'].astype(str)
clicks_pd['audienceid'] = clicks_pd['audienceid'].astype(str)
sales_funnel['audienceid'] = sales_funnel['audienceid'].astype(str)
new_scores_pd['audienceid'] = new_scores_pd['audienceid'].astype(str)

full_fan_with_agg = (
  new_scores_pd
    .merge(agg_fields_pd, on='audienceid', how='left')
    .merge(clicks_pd, on='audienceid', how='left')
    .merge(sent_pd, on='audienceid', how='left')
    .merge(sales_funnel, on='audienceid', how='left')
  )
full_fan_with_agg['sales_funnel'] = full_fan_with_agg['sales_funnel'].fillna('Null')
full_fan_with_agg = full_fan_with_agg.fillna(0)

full_fan_with_agg['click_rate'] = (full_fan_with_agg['total_clicks'] / full_fan_with_agg['total_sent']).replace([float('inf')], 0).fillna(0).round(4)

display(full_fan_with_agg)
  